<a href="https://colab.research.google.com/github/springboardmentor123g/PlantDocBot/blob/intern-AnshikaSahu/Text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import numpy as np
import evaluate


In [ ]:
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")
df = df.drop(columns=["image"])
df = df.explode("captions")
df = df.rename(columns={"captions": "text"})

label_col = "caption"
text_col = "text"

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df[label_col].tolist())
num_labels = len(encoder.classes_)

df_train, df_test = train_test_split(
    df, train_size=0.8, random_state=42, stratify=df["label"]
)
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(data):
    return tokenizer(data[text_col], truncation=True)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

results = trainer.evaluate()
print("Final evaluation:", results)

trainer.save_model("./best_plant_text_classifier")
tokenizer.save_pretrained("./best_plant_text_classifier")

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

encoder = LabelEncoder()
encoder.fit(df[label_col].tolist())
np.save("/content/drive/MyDrive/encoder_classes.npy", encoder.classes_)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
model.save_pretrained("/content/drive/MyDrive/best_plant_text_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/best_plant_text_classifier")


In [ ]:
import os
os.kill(os.getpid(), 9)



In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

repo_path = "/content/drive/MyDrive/best_plant_text_classifier"
branch_name = "intern-AnshikaSahu"
notebook_file = "Text_classification.ipynb"
commit_message = "Updated notebook with new changes"

os.chdir(repo_path)
print("Current folder:", os.getcwd())

os.system(f"git checkout -b {branch_name} || git checkout {branch_name}")
os.system(f"git add {notebook_file}")
os.system(f'git commit -m "{commit_message}"')
os.system(f"git push origin {branch_name}")

print(f"Notebook pushed successfully to branch '{branch_name}'!")


In [ ]:
!pip install --upgrade transformers
!pip install --upgrade huggingface-hub
!pip install safetensors


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp -r "/content/drive/MyDrive/best_plant_text_classifier" "/content/best_plant_text_classifier"



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "/content/best_plant_text_classifier"

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

print("Model loaded!")


In [ ]:

encoder = LabelEncoder()
encoder.classes_ = np.load("/content/drive/MyDrive/encoder_classes.npy", allow_pickle=True)

sample_text = "leaf has holes in between"

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    predicted_class_index = outputs.logits.argmax(-1).item()

predicted_label = encoder.inverse_transform([predicted_class_index])[0]

print(f" Input text: {sample_text}")
print(f" Predicted label: {predicted_label}")
